# Mamba-TKAN TEC Reconstruction — Google Colab Training

This notebook trains the Physics-Guided Hybrid Mamba-TKAN Framework on Google Colab using a T4 or A100 GPU. All checkpoints, logs, and exports are automatically saved to Google Drive.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Define Project Path
PROJECT_DIR = '/content/drive/MyDrive/TEC_Project'

import os
if not os.path.exists(PROJECT_DIR):
    print(f"\nWARNING: {PROJECT_DIR} not found.")
    print("Please upload the project folder to Google Drive before continuing.")
else:
    %cd {PROJECT_DIR}
    print(f"\nWorking directory changed to: {os.getcwd()}")

In [ ]:
# 3. Install Requirements & Verify Environment
!pip install -q -r requirements_colab.txt

import torch
from utils.env_config import configure, print_environment_summary

# Configure paths (routes outputs to Drive)
paths = configure(drive_root=PROJECT_DIR)
print_environment_summary()

assert torch.cuda.is_available(), "CUDA is not available. Go to Runtime > Change runtime type > GPU."

In [ ]:
# 4. Launch TensorBoard (Optional, runs in background)
%load_ext tensorboard
%tensorboard --logdir {paths.tensorboard} --port 6006

In [ ]:
# 5. Start / Resume Training
# The script will automatically resume from latest.pt if it exists in checkpoints/

!python train.py \
  --epochs 150 \
  --batch-size 64 \
  --lr 1e-4 \
  --device cuda \
  --experiment-name "colab_run_01" \
  --mode production \
  --colab \
  --drive-root {PROJECT_DIR} \
  --export

In [ ]:
# 6. Verify Exports
import os

print("\n--- Exported Models ---")
export_dir = str(paths.exports)
if os.path.exists(export_dir):
    for f in sorted(os.listdir(export_dir)):
        size_mb = os.path.getsize(os.path.join(export_dir, f)) / (1024 * 1024)
        print(f"{f}  ({size_mb:.2f} MB)")
else:
    print("No exports found.")